# ASX OHLCV Lake Check

This notebook inspects the ASX200 lake flow in MinIO. It validates the request config, counts raw and conformed objects, reads the curated parquet panel, and writes a compact summary artifact.

In [ ]:
from __future__ import annotations

import io
import json
from datetime import datetime, timezone
from pathlib import Path
from urllib import request

import boto3
import pandas as pd
import pyarrow.parquet as pq

CONFIG_PATH = Path("/home/jovyan/config/asx/asx_data_request.json")
OUTPUT_PATH = Path("/home/jovyan/data/asx_ohlcv_summary.json")
MINIO_ENDPOINT = "http://minio:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
MINIO_REGION = "local-01"
RAW_BUCKET = "raw"
CONFORMED_BUCKET = "conformed"
CURATED_BUCKET = "curated"
TRINO_URL = "http://trino:8080/v1/statement"
TRINO_SQL = (
    "SELECT ticker, run_date, row_count, start_trade_date, end_trade_date, min_close, max_close, latest_close, average_volume, generated_at "
    "FROM demo.asx_ohlcv_summary ORDER BY ticker"
)

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"ASX request config was not found at canonical notebook path {CONFIG_PATH}.")

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
dataset_id = str(config["dataset_id"])
exchange = str(config["exchange"])
prefix = f"tabular/{dataset_id}/exchange={exchange}/"
curated_key = f"{prefix}asx_ohlcv_panel_curated.parquet"

s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name=MINIO_REGION,
)

print(f"Config path: {CONFIG_PATH}")
print(f"Lake prefix: {prefix}")
config


In [ ]:
def list_keys(bucket: str, prefix: str) -> list[str]:
    keys = []
    token = None
    while True:
        kwargs = {"Bucket": bucket, "Prefix": prefix, "MaxKeys": 1000}
        if token:
            kwargs["ContinuationToken"] = token
        response = s3.list_objects_v2(**kwargs)
        keys.extend(obj["Key"] for obj in response.get("Contents", []))
        if not response.get("IsTruncated"):
            break
        token = response.get("NextContinuationToken")
    return keys


def run_trino_query(sql: str) -> list[dict[str, object]]:
    headers = {"X-Trino-User": "jovyan", "X-Trino-Catalog": "demo", "X-Trino-Schema": "demo"}
    req = request.Request(TRINO_URL, data=sql.encode("utf-8"), headers=headers, method="POST")
    rows = []
    columns = []

    with request.urlopen(req, timeout=30) as response:
        payload = json.loads(response.read().decode("utf-8"))

    while True:
        if not columns and payload.get("columns"):
            columns = [column["name"] for column in payload["columns"]]
        rows.extend(payload.get("data", []))
        next_uri = payload.get("nextUri")
        if not next_uri:
            break
        with request.urlopen(next_uri, timeout=30) as response:
            payload = json.loads(response.read().decode("utf-8"))

    return [dict(zip(columns, row)) for row in rows]

raw_keys = [key for key in list_keys(RAW_BUCKET, prefix) if key.endswith(".csv")]
conformed_keys = [key for key in list_keys(CONFORMED_BUCKET, prefix) if key.endswith(".parquet")]
curated_keys = [key for key in list_keys(CURATED_BUCKET, prefix) if key.endswith(".parquet")]

lake_summary = {
    "expected_ticker_count": len(config["ticker_list"]),
    "raw_object_count": len(raw_keys),
    "conformed_object_count": len(conformed_keys),
    "curated_object_count": len(curated_keys),
    "sample_raw_keys": raw_keys[:3],
    "sample_conformed_keys": conformed_keys[:3],
    "sample_curated_keys": curated_keys[:3],
}
lake_summary


In [ ]:
curated_response = s3.get_object(Bucket=CURATED_BUCKET, Key=curated_key)
curated_table = pq.read_table(io.BytesIO(curated_response["Body"].read()))
df = curated_table.to_pandas()
df["trade_date"] = pd.to_datetime(df["trade_date"], errors="coerce")
df = df.dropna(subset=["trade_date"]).sort_values(["ticker", "trade_date"]).reset_index(drop=True)
df["close"] = pd.to_numeric(df["close"], errors="coerce")
df["volume"] = pd.to_numeric(df["volume"], errors="coerce")
df["daily_return"] = df.groupby("ticker")["close"].pct_change()

dataset_summary = {
    "rows": int(len(df)),
    "tickers": int(df["ticker"].nunique()),
    "date_min": df["trade_date"].min().date().isoformat(),
    "date_max": df["trade_date"].max().date().isoformat(),
    "null_close_rows": int(df["close"].isna().sum()),
    "null_volume_rows": int(df["volume"].isna().sum()),
}

latest_close = (
    df.groupby("ticker", as_index=False)
    .tail(1)
    .loc[:, ["ticker", "trade_date", "close", "daily_return", "volume"]]
    .sort_values("ticker")
)

trino_rows = []
trino_error = None
try:
    trino_rows = run_trino_query(TRINO_SQL)
except Exception as exc:
    trino_error = str(exc)

dataset_summary


In [ ]:
output_payload = {
    "generated_at": datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z"),
    "config_path": str(CONFIG_PATH),
    "config": config,
    "lake_summary": lake_summary,
    "dataset_summary": dataset_summary,
    "latest_close_records": latest_close.to_dict(orient="records"),
    "trino_query": TRINO_SQL,
    "trino_row_count": len(trino_rows),
    "trino_error": trino_error,
    "trino_rows": trino_rows,
}

OUTPUT_PATH.write_text(json.dumps(output_payload, indent=2, sort_keys=True, default=str), encoding="utf-8")
print(f"Summary written to {OUTPUT_PATH}")
output_payload
